In [1]:
from pathlib import Path
from data import ReconstructionDataset
from cgan import StateReconstructor
from rho import rho_from_params

root = Path.cwd().parent.parent
ds = ReconstructionDataset(f"{root}/data/train_noisy_35.h5")

In [2]:
import torch
from torch.utils.data import DataLoader
from torch import nn, optim

device = "cuda" if torch.cuda.is_available() else "cpu"

loader = DataLoader(ds, batch_size=5, shuffle=True)
model = StateReconstructor().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()


In [3]:
epochs = 4
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for w, rho_true in loader:
        w = w.to(device)
        rho_true = rho_true.to(device)

        params = model(w)
        rho_hat = rho_from_params(params, N=64)

        loss = loss_fn(rho_hat.real, rho_true.real) + loss_fn(rho_hat.imag, rho_hat.imag)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * w.size(0)

    print(f"epoch {epoch + 1}: loss = {epoch_loss / len(ds):.6f}")

epoch 0: loss = 0.000194
epoch 1: loss = 0.000193
epoch 2: loss = 0.000192
epoch 3: loss = 0.000191
